In [1]:
%%time
# ## 測時間
# ### 結果為正 表示測試的電腦比惜物網"快" n 秒
# ### 結果為負 表示測試的電腦比惜物網"慢" n 秒
import requests,json,time
main_url='https://shwoo.gov.taipei/shwoo'
time.time()-(int(json.loads(requests.post(main_url+'/product/product00/websocket', data={'message':'{"no":4}'}).json())['serverTime'])/1000)


Wall time: 375 ms


-3.9810006618499756

In [ ]:
%%time
### Login ###
import datetime
from datetime import datetime
import requests,re,os,json
import time
from bs4 import BeautifulSoup

import numpy as np
import keras
import sys

 
import shutil
from requests import Session
import PIL.Image as Image
import win32clipboard,win32con

import io
def copy_toClipboard(text):
    win32clipboard.OpenClipboard()
    win32clipboard.EmptyClipboard()
    win32clipboard.SetClipboardData(win32con.CF_UNICODETEXT, text)
    win32clipboard.CloseClipboard()
def ReadClipboard():
    #拿剪貼簿裡面的東西，就同等於按下 Control + V
    win32clipboard.OpenClipboard() 
    data =""
    if win32clipboard.IsClipboardFormatAvailable(win32clipboard.CF_DIB):
        data = win32clipboard.GetClipboardData(win32clipboard.CF_DIB)
    win32clipboard.CloseClipboard() 
    try:
        image = Image.open(io.BytesIO(data))
        image.save('img.jpeg')
        return "OK"
    except:
        return "wait"

bid_model=keras.models.load_model('model/bid_model.h5')
characters = '0123456789abcdefghijklmnopqrstuvwxyz'
def bid_price(auid, n_times=1):
    ### bidding price ###
    home_url=main_url+'/home/home00/index'
    req_session=Session()
    cookie = '; '.join(['='.join(s) for s in req_session.get(home_url).cookies.items()])
    hds={
        'Accept': 'application/json, text/javascript, */*; q=0.01',
        'Accept-Encoding': 'gzip, deflate, br',
        'Accept-Language': 'zh-TW,zh;q=0.9,en-US;q=0.8,en;q=0.7',
        'Connection': 'keep-alive',
        'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
        'Host': 'shwoo.gov.taipei',
        'Origin': 'https://shwoo.gov.taipei',
        'Referer': main_url+'/home/home00/index',
        'Cookie': cookie,
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_12_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/80.0.3987.163 Safari/537.36',
    }
    hds['Referer']=main_url+'/product/product00/product?AUID='+auid
    pl={ 'message': '{"no":1,"auid":"%s"}'%auid }
    new_price_json=json.loads(requests.post(main_url+'/product/product00/websocket', data=pl, timeout=0.8).json())
    price_now = new_price_json['money']
    return price_now
    #####################
##############################################################################################
def check_first_bidder(account,auid):
    for i in range(0,10):
        try:
            first_account = json.loads(requests.post(main_url+'/product/product00/websocket', data={'message':'{"no":3,"auid":"%s"}'%auid,}, timeout=0.8).json())['detail'][0]
        except: print('check timeout');continue
        if first_account: break
    if '得標' in first_account['account']: return 'end'
    if (account[:-2]+"**") not in first_account['account']:
        print('[NOT YOU]', first_account['account'], first_account['bidprice'])
    else: print(first_account['account'], first_account['bidprice']);return 1
##############################################################################################

def processPic_login(filepath):
    img = Image.open(filepath)
    img = img.resize((85,27))
    im = np.array(img)
    X = np.zeros((1, 27,85, 3), dtype=np.float32)
    X[0] = im / 255.0
    return X

def processPic_bid(filepath):
    img = Image.open(filepath).crop((0,0,120,35))
#     from matplotlib.pyplot import imshow
#     imshow(img)
    img = img.resize((120,35))
    im = np.array(img)
    X = np.zeros((1, 35,120, 3), dtype=np.float32)
    X[0] = im / 255.0
    return X
account  = input('02.帳　　號:　') 
auid = input('01.投標物件: ')
nowPrice = bid_price(auid)
check_first_bidder(account,auid)
maxPrice  = input('02.目前價格$%s，最高投標金額: '% nowPrice)


while True:
    if int(nowPrice) >= int(maxPrice):
        print ('超過最高投標金額')
        break
    try:
        ReadClipResult = ReadClipboard()
        if ReadClipResult == "OK":
            X = processPic_bid('img.jpeg')
            predicted = ''.join([characters[(np.argmax(p))] for p in bid_model.predict(X)])
            print(predicted)
            copy_toClipboard(predicted)
        time.sleep(0.01)
    except:
        pass